## 13.06 子词嵌入


### 环境配置


In [1]:
import logging
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    import torch_npu

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()
import math
import re
import collections


### 练习 13.6.1

**题目：** 例如，英语中大约有 $3\times 10^8$ 种可能的 $6$-元组。子词太多会有什么问题呢？如何解决这个问题？提示：请参阅 fastText 论文第 3.2 节末尾。

**解答：** 子词数量过大会带来三个问题：

- **存储开销大**：为每个子词维护一个嵌入向量，$3\times 10^8$ 个子词对应的嵌入表将占据海量内存；
- **稀疏性**：大量子词在语料中几乎不出现，其嵌入得不到充分训练；
- **计算开销**：每次前向都要查表、聚合，子词表越大越慢。

fastText 论文第 3.2 节末尾的解决方案是**哈希（hashing trick）**：用一个固定大小（如 $10^7$）的哈希表，把所有 n-gram 子词哈希映射到 $[0, B)$ 的桶中，每个桶对应一个嵌入向量。哈希冲突（不同子词落入同一桶）由多个子词共享向量来吸收，代价是轻微的信息混叠，但大大压缩了模型大小。

下面用一个小实验演示哈希桶对“子词表大小”的压缩效果：


In [2]:
def hashing_trick(grams, B=100):
    buckets = [hash(g) % B for g in grams]
    return buckets

words = ['where', 'here', 'there']
grams = []
for w in words:
    grams += ['<'+w[:i]+'>' for i in range(1, len(w) + 1)]
print(f'子词总数: {len(grams)}, 去重后: {len(set(grams))}')
for B in (10, 50, 100, 1000):
    print(f'B={B:5d}: 占用桶数 {len(set(hashing_trick(grams, B)))}')


子词总数: 14, 去重后: 14
B=   10: 占用桶数 9
B=   50: 占用桶数 13
B=  100: 占用桶数 14
B= 1000: 占用桶数 14


### 练习 13.6.2

**题目：** 如何在连续词袋模型的基础上设计一个子词嵌入模型？

**解答：** 思路与 fastText 一致，把“整词嵌入”替换为“子词嵌入之和”：

1. 对词表 $\mathcal{V}$ 中每个词 $w$，预先计算其 n-gram 子词集合 $\mathcal{G}_w$（如 3~6 元组，加上 `\<w\>` 与 `\<\/w\>` 边界标记）；
2. 为每个子词 $g$ 维护嵌入 $\mathbf{z}_g$，词的表示定义为子词嵌入之和：

$$\mathbf{v}_w = \sum_{g \in \mathcal{G}_w} \mathbf{z}_g;$$

3. 上下文词向量同样由子词嵌入聚合得到（或直接维护整词上下文嵌入）；
4. 在 CBOW 中，把“上下文词向量的均值”替换为“上下文词各自子词向量之和的均值”，损失仍用负采样或层序 softmax。

这样做的好处：形态相近的词（如 “walk”、“walking”、“walked”）共享大部分子词，彼此的词向量天然接近；未登录词（OOV）也能由子词拼出表示，解决了整词模型无法处理 OOV 的问题。


### 练习 13.6.3

**题目：** 要获得大小为 $m$ 的词表，当初始符号词表大小为 $n$ 时，需要多少合并操作？

**解答：** 最多需要 $m - n$ 次合并。

BPE 的每次合并把**两个相邻符号**替换为**一个新符号**：符号总数从 $n$ 变为 $n-1$（少 2 个多 1 个）。要从 $n$ 达到 $m$ 个符号，需要 $m - n$ 次合并。

注意这是“最多”需要：若训练语料中可用配对不足，合并可能在达到 $m$ 之前终止（此时实际词表小于 $m$）；反之若要求恰好 $m$，则必须执行 $m-n$ 次。


### 练习 13.6.4

**题目：** 如何扩展字节对编码的思想来提取短语？

**解答：** 把 BPE 的合并对象从“字符/子词”换成“词/短语”即可：

1. 先在语料上做标准分词，得到词序列；
2. 统计相邻两个词组成候选短语的出现频率；
3. 把最高频的相邻词对合并为一个新“短语符号”（如 “new york” → “new_york”），替换语料中所有出现；
4. 重复第 2–3 步直到达到目标短语数或没有超过阈值频率的配对。

这与 BPE 的迭代合并框架完全相同，只是基本单位从字符变成了词。word2vec 论文第四节（13.1.2 练习）中训练短语向量的“特殊标记”做法，本质就是这里用 BPE 自动发现固定短语的过程。


---

## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
